In [0]:
%run ../../config/utils

In [0]:
import pyspark.sql.functions as f
import pandas as pd
import numpy as np
import mlflow
from mlflow.client import MlflowClient
from mlflow.models.signature import infer_signature
from datetime import datetime
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, roc_curve, auc, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import json


mlflow.set_registry_uri('databricks-uc')
mlflow.autolog(disable=True)

In [0]:
run_date_str = dbutils.widgets.get('run_as_date')
current_year = int(run_date_str[:4])
TENURE_GROUP = dbutils.widgets.get('tenure_group') if dbutils.widgets.get('tenure_group') in ['tenured', 'new'] else 'tenured'
gm_preprocessing_uri = f"models:/{gm_preprocessing_catalog}@{TENURE_GROUP}"
preprocess = mlflow.sklearn.load_model(gm_preprocessing_uri)

In [0]:
latest_run_date = spark.table(gm_dataset).filter(f.col('RUN_DATE') <= run_date_str).agg(f.max('RUN_DATE').alias('max_dt')).first()['max_dt']
data_spark = spark.table(gm_dataset).filter((f.col('RUN_DATE') == latest_run_date) & (f.col('TENURE_GROUP_FILTER') == TENURE_GROUP)).drop('RUN_DATE', 'TENURE_GROUP_FILTER')
data_pd_v3 =data_spark.toPandas()


data_pd_v3.columns = data_pd_v3.columns.str.upper()
y = data_pd_v3["GM_PURCHASE"]

print("Starting to preprocess input data....shape:",data_pd_v3.shape)

#preprocess.fit(df_sample)

processed = preprocess.fit_transform(data_pd_v3)

if np.array(processed).ndim == 0:
    X_pre = processed.toarray()
else:
    X_pre = np.array(processed)

print("Created Preprocessing file")


#y_pre = np.array(y.values.reshape(len(y), 1))

data_pd_v3_pre = pd.DataFrame(X_pre)
data_pd_v3_pre.columns = data_pd_v3_pre.columns.astype(str)

In [0]:
X_train, X_test, y_train, y_test = train_test_split(data_pd_v3_pre, y,stratify=y, test_size=0.2, random_state=42)

# Hyperparameter Tuning

In [0]:
xgb_model = XGBClassifier(eval_metric='logloss')

In [0]:
param_grid = {
    "n_estimators": [None, 50, 100, 150, 200],
    "learning_rate": [None, 0.01, 0.05, 0.1, 0.2],
    "max_depth": [None, 3, 4, 5, 6],
    "gamma": [None, 0, 0.1, 0.2],
    "reg_alpha": [None, 0, 0.01, 0.1, 0.2],
    "reg_lambda": [None, 1, 1.5, 2.0],
    "subsample": [None, 0.6, 0.8, 1.0],
    "colsample_bytree": [None, 0.6, 0.8, 1.0],
}


In [0]:
param_grid = {'n_estimators': [50,100,150,200,None],
             'learning_rate': [0.01, 0.05, 0.1, 0.2,None],
             'max_depth': [3, 4, 5, 6,None]}

In [0]:
grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, cv=3, scoring='accuracy', verbose=1, n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best accuracy: ", grid_search.best_score_)

In [0]:
param_grid = {'n_estimators': [grid_search.best_params_['n_estimators'],None],
    'learning_rate': [grid_search.best_params_['learning_rate'],None],
              'max_depth': [grid_search.best_params_['max_depth'], grid_search.best_params_['max_depth']+1,None],
              'gamma': [0, 0.1, 0.2,None],
                  'reg_alpha': [0, 0.01, 0.1, 0.2,None],
    'reg_lambda': [1, 1.5, 2.0,None]
             }

In [0]:
grid_search_2 = GridSearchCV(estimator=xgb_model, param_grid=param_grid, cv=3, scoring='accuracy', verbose=1, n_jobs=-1)
grid_search_2.fit(X_train, y_train)

print("Best parameters found: ", grid_search_2.best_params_)
print("Best accuracy: ", grid_search_2.best_score_)

In [0]:
param_grid = {'n_estimators': [grid_search_2.best_params_['n_estimators'],None],
    'learning_rate': [grid_search_2.best_params_['learning_rate'],None],
              'max_depth': [grid_search_2.best_params_['max_depth'],None],
              'gamma': [0, 0.1, 0.2,None],
                  'reg_alpha': [grid_search_2.best_params_['reg_alpha'],None],
    'reg_lambda': [grid_search_2.best_params_['reg_lambda'], grid_search_2.best_params_['reg_lambda'] + 1, None],
                  'subsample': [0.6, 0.8, 1.0,None],
    'colsample_bytree': [0.6, 0.8, 1.0,None],
             }

In [0]:
grid_search_3 = GridSearchCV(estimator=xgb_model, param_grid=param_grid, cv=3, scoring='accuracy', verbose=1, n_jobs=-1)
grid_search_3.fit(X_train, y_train)

print("Best parameters found: ", grid_search_3.best_params_)
print("Best accuracy: ", grid_search_3.best_score_)

In [0]:
best_accuracy = -1
for grid in [grid_search, grid_search_2, grid_search_3]:
    if grid.best_score_ > best_accuracy:
        best_accuracy = grid.best_score_
        best_grid = grid.best_params_
print(f'Final Best Accuracy = {best_accuracy}')
print(f'Final Best Parameters = {best_grid}')

## Fit Best Model to training

In [0]:
experiment_name = experiment_name_gm_model

mlflow.xgboost.autolog(disable=False, log_input_examples=True, log_models=False, log_datasets=False)

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri('databricks-uc')

if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(name=experiment_name) 
mlflow.set_experiment(experiment_name)

feature_dataset = mlflow.data.from_spark(data_spark, name = 'gm_dataset')

In [0]:
%sh
mkdir tmp

In [0]:
mark_datetime = datetime.strftime(datetime.now(), '_%Y-%m-%d_%H-%M-%S')
with mlflow.start_run(run_name=f'training_gm_model_{TENURE_GROUP}_{mark_datetime}') as run:
    mlflow.log_input(feature_dataset, context="source")
    mlflow.log_input(mlflow.data.from_pandas(X_train, source=feature_dataset.source), context="training")
    mlflow.log_input(mlflow.data.from_pandas(X_test, source=feature_dataset.source), context="testing")

    # fit model no training data
    model = XGBClassifier(**best_grid)
    model.fit(X_train, y_train)

    y_pred_test = model.predict(X_test)
    print(classification_report(y_test, y_pred_test))
    with open("./tmp/classification_report_test.txt", "w") as text_file:
        text_file.write(classification_report(y_test, y_pred_test))
    mlflow.log_artifact("./tmp/classification_report_test.txt")

    y_pred_train = model.predict(X_train)

    with open("./tmp/classification_report_train.txt", "w") as text_file:
        text_file.write(classification_report(y_train, y_pred_train))
    mlflow.log_artifact("./tmp/classification_report_train.txt")

    test_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    mlflow.log_metric("test_auc", test_auc)
    mlflow.log_metric("train_auc", train_auc)
    accuracy_score(y_test, y_pred_test)
    accuracy_score(y_train, y_pred_train)
    recall_score(y_test, y_pred_test)
    recall_score(y_train, y_pred_train)

    plt.figure(figsize=(7, 5))
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test)[:, 1])
    auc_val = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, label=f'Test AUC = {auc_val:.3f}', linewidth=2)
    plt.plot([0, 1], [0, 1], 'r--', label='Random guess', linewidth=1)
    plt.xlim(0, 1); plt.ylim(0, 1)
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend(loc='lower right')
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig('./tmp/roc_curve_test.png', dpi=150)
    mlflow.log_artifact('./tmp/roc_curve_test.png')
    plt.close()


    classes = np.unique(np.concatenate([y_test, y_pred_test]))  # ensures consistent order
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred_test,
        labels=classes,
        display_labels=classes,
        cmap=None,           # use default
        values_format='d',   # integers for counts
        ax=ax
    )
    ax.set_title("Confusion Matrix - Test")
    plt.tight_layout()
    plt.savefig("./tmp/confusion_matrix_test.png", dpi=150)
    plt.close()
    mlflow.log_artifact('./tmp/confusion_matrix_test.png')


    score = model.predict_proba(X_test)

    tenured_eval_data = pd.DataFrame()
    tenured_eval_data['MBRSHP_SID']= list(data_pd_v3.iloc[list(X_test.index)]['MBRSHP_SID'])
    tenured_eval_data['Probability'] = score[:,1]
    tenured_eval_data['Actuals'] = list(y_test)
    tenured_eval_data['Decile'] = 10 - pd.qcut(tenured_eval_data['Probability'].rank(method='first'), 10, labels = False)

    tenured_eval_data.groupby('Decile').agg({'MBRSHP_SID':'nunique','Actuals':'sum','Probability':'min'}).to_csv('./tmp/tenured_eval_data.csv')
    mlflow.log_artifact('./tmp/tenured_eval_data.csv')


    signature = infer_signature(X_train.sample(5), y_pred_test[:5])
 

    model_info = mlflow.xgboost.log_model(
        xgb_model=model,
        artifact_path="model",
        signature=signature,
        model_format='json',
        registered_model_name=gm_model_catalog,
    )
    


In [0]:
client = MlflowClient()
client.set_registered_model_alias(name=gm_model_catalog, alias=TENURE_GROUP, version=model_info.registered_model_version)

In [0]:
%sh
rm -r tmp